# Chapter 7 Lab — Substrate Independence Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/liquid-books/basal-cognition/blob/main/notebooks/ch07-lab-substrate-sim.ipynb)

**Basal Cognition · Dr. Ernesto Lee**

---

## What you will build

The same goal-seeking algorithm — minimize distance to a target — running in three different 'substrates':

1. **Cell substrate**: a grid of agents that move using a simple chemical gradient rule (like xenobot cilia)
2. **Robot substrate**: a swarm of agents that move physically through a 2D space with obstacles
3. **Comparison**: side-by-side plots showing convergence speed and obstacle-handling for each

The goal is the same. The substrate changes. You will observe how the same organizational principle plays out differently in different media — and where each breaks.

**Estimated time:** 45–60 minutes

**No prior Python experience needed.** Every block has a comment explaining what it does.

---

## The key idea

Substrate independence (weak version): the *organizational principle* of goal-seeking behavior — detect deviation from target, take corrective action — can run on multiple substrates. What changes is efficiency, robustness to obstacles, and failure modes. This simulation makes that visible.

In [ ]:
# Install dependencies (only needed if running outside Colab)
# In Colab, matplotlib and numpy are already available.
%pip install -q matplotlib numpy

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set random seed for reproducibility
np.random.seed(42)

print('Imports complete.')

## Part 1: The Shared Goal

Both 'substrates' pursue the same goal: move agents toward a target position.
The agents start scattered randomly. The target is fixed.
Success = all agents within a threshold distance of the target.

In [ ]:
# Shared configuration
N_AGENTS = 20          # Number of agents in each substrate
TARGET = np.array([0.8, 0.8])   # Target position (x, y) in [0, 1] space
SUCCESS_RADIUS = 0.05  # Agents within this distance count as 'arrived'
MAX_STEPS = 200        # Maximum simulation steps

def scatter_agents(n=N_AGENTS):
    """Randomly scatter agents in [0,1] x [0,1] space, away from the target."""
    positions = np.random.uniform(0, 0.4, size=(n, 2))  # Start in lower-left
    return positions

def count_arrived(positions, target=TARGET, radius=SUCCESS_RADIUS):
    """Count agents within SUCCESS_RADIUS of target."""
    distances = np.linalg.norm(positions - target, axis=1)
    return np.sum(distances < radius)

print(f'Goal: move {N_AGENTS} agents to target at {TARGET}')
print(f'Success: agents within radius {SUCCESS_RADIUS} of target')

## Part 2: Cell Substrate

**Rule:** Each agent moves toward the target at a rate proportional to
the distance (like a cell following a chemical gradient — stronger signal closer to source).
Add small random noise to simulate Brownian motion.

This mimics how xenobot cilia collectively orient toward a gradient without any
individual cell 'knowing' the full picture.

In [ ]:
def run_cell_substrate(n_agents=N_AGENTS, max_steps=MAX_STEPS, step_size=0.04, noise=0.008):
    """
    Cell substrate: gradient-following with Brownian noise.
    Returns history of positions at each step and arrival counts.
    """
    positions = scatter_agents(n_agents)
    history = [positions.copy()]
    arrivals = [count_arrived(positions)]

    for step in range(max_steps):
        # Direction toward target for each agent
        directions = TARGET - positions
        norms = np.linalg.norm(directions, axis=1, keepdims=True)
        norms = np.where(norms < 1e-6, 1e-6, norms)  # Avoid division by zero
        unit_directions = directions / norms

        # Move toward target + Brownian noise
        positions = positions + step_size * unit_directions + np.random.normal(0, noise, positions.shape)
        positions = np.clip(positions, 0, 1)  # Keep in bounds

        history.append(positions.copy())
        arrivals.append(count_arrived(positions))

    return history, arrivals

cell_history, cell_arrivals = run_cell_substrate()
print(f'Cell substrate: {cell_arrivals[-1]}/{N_AGENTS} agents arrived after {MAX_STEPS} steps')

## Part 3: Robot Substrate

**Rule:** Each agent moves toward the target with a fixed step size.
But there is an **obstacle** — a wall blocking the direct path.
Agents must detect the obstacle and route around it.

The goal is identical. The obstacle changes the path, not the destination.

In [ ]:
# Define a rectangular obstacle
OBSTACLE = {'x': 0.45, 'y': 0.3, 'w': 0.1, 'h': 0.45}  # x, y, width, height

def in_obstacle(pos, obs=OBSTACLE):
    """Check if a position is inside the obstacle."""
    return (obs['x'] <= pos[0] <= obs['x'] + obs['w'] and
            obs['y'] <= pos[1] <= obs['y'] + obs['h'])

def run_robot_substrate(n_agents=N_AGENTS, max_steps=MAX_STEPS, step_size=0.025, noise=0.005):
    """
    Robot substrate: fixed step toward target, avoid obstacle.
    When blocked, agents add a perpendicular deflection to route around.
    """
    positions = scatter_agents(n_agents)
    history = [positions.copy()]
    arrivals = [count_arrived(positions)]

    for step in range(max_steps):
        new_positions = positions.copy()
        for i, pos in enumerate(positions):
            direction = TARGET - pos
            norm = np.linalg.norm(direction)
            if norm < 1e-6:
                continue
            unit = direction / norm
            candidate = pos + step_size * unit + np.random.normal(0, noise, 2)
            candidate = np.clip(candidate, 0, 1)

            # If the candidate position is in the obstacle, deflect perpendicular
            if in_obstacle(candidate):
                perp = np.array([-unit[1], unit[0]])  # Rotate 90 degrees
                candidate = pos + step_size * perp + np.random.normal(0, noise, 2)
                candidate = np.clip(candidate, 0, 1)

            new_positions[i] = candidate

        positions = new_positions
        history.append(positions.copy())
        arrivals.append(count_arrived(positions))

    return history, arrivals

robot_history, robot_arrivals = run_robot_substrate()
print(f'Robot substrate: {robot_arrivals[-1]}/{N_AGENTS} agents arrived after {MAX_STEPS} steps')

## Part 4: Compare the Substrates

Same goal. Different substrates. Different convergence profiles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Final positions, cell substrate ---
ax = axes[0]
final_cell = cell_history[-1]
ax.scatter(final_cell[:, 0], final_cell[:, 1], c='steelblue', s=40, label='Agents', zorder=3)
ax.scatter(*TARGET, c='limegreen', s=200, marker='*', label='Target', zorder=4)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Cell Substrate: Final Positions', fontsize=12)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend()
ax.set_aspect('equal')

# --- Plot 2: Final positions, robot substrate ---
ax = axes[1]
final_robot = robot_history[-1]
ax.scatter(final_robot[:, 0], final_robot[:, 1], c='coral', s=40, label='Agents', zorder=3)
ax.scatter(*TARGET, c='limegreen', s=200, marker='*', label='Target', zorder=4)
# Draw obstacle
obs = OBSTACLE
rect = patches.Rectangle((obs['x'], obs['y']), obs['w'], obs['h'],
                           linewidth=1, edgecolor='gray', facecolor='lightgray', zorder=2)
ax.add_patch(rect)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Robot Substrate: Final Positions (with obstacle)', fontsize=12)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend()
ax.set_aspect('equal')

# --- Plot 3: Convergence over time ---
ax = axes[2]
steps = list(range(MAX_STEPS + 1))
ax.plot(steps, cell_arrivals, color='steelblue', linewidth=2, label='Cell substrate')
ax.plot(steps, robot_arrivals, color='coral', linewidth=2, label='Robot substrate')
ax.axhline(y=N_AGENTS, color='limegreen', linestyle='--', alpha=0.7, label='All arrived')
ax.set_xlabel('Step')
ax.set_ylabel('Agents at target')
ax.set_title('Convergence: Same Goal, Different Substrates', fontsize=12)
ax.legend()
ax.set_ylim(0, N_AGENTS + 1)

plt.tight_layout()
plt.savefig('ch07-substrate-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved as ch07-substrate-comparison.png')

## Part 5: Your Turn (TODO)

Now modify the simulation to answer these questions. Each one has a TODO block below.

In [ ]:
# TODO 1: Change the target location
# Move the target to [0.2, 0.9] and re-run both simulations.
# Does the obstacle block the cell substrate now?
# Which substrate converges faster when the obstacle is in the path?

NEW_TARGET = np.array([0.2, 0.9])  # TODO: Try different positions

# YOUR CODE HERE
# Hint: create new run functions or modify the TARGET variable above and re-run cells
print('TODO 1: Modify the target and observe the effect on both substrates.')

In [ ]:
# TODO 2: Add a second obstacle
# Add a second rectangular obstacle that blocks the robot substrate's escape route.
# What happens? Does the robot substrate get trapped?
# Does the cell substrate (with its gradient + noise) handle this differently?

OBSTACLE_2 = {'x': 0.2, 'y': 0.5, 'w': 0.15, 'h': 0.35}  # TODO: Experiment with position

# YOUR CODE HERE
# Hint: modify in_obstacle() to check both obstacles
print('TODO 2: Add a second obstacle and observe failure modes in each substrate.')

In [ ]:
# TODO 3: Remove half the agents mid-run (damage experiment)
# Stop both simulations at step 50. Remove 10 agents from each (simulate damage).
# Resume for another 150 steps.
# Does the remaining population still reach the target?
# Does removing agents affect convergence speed?

# YOUR CODE HERE
# Hint: run for 50 steps, slice positions to keep only N_AGENTS//2 rows,
# then continue the loop
print('TODO 3: Damage experiment — remove agents mid-run and observe robustness.')

## Deliverable

Answer the following in a markdown cell below this one (double-click to edit):

1. **What changed between the cell and robot substrates?** Not the goal — the goal was identical. What specific properties of each substrate produced different convergence behavior?

2. **Which substrate was more robust to damage (TODO 3)?** Why do you think that is? Connect your answer to the chapter's discussion of setpoints and error signals.

3. **What would a 'software agent' substrate look like in this simulation?** Propose a third run function — `run_software_substrate()` — with a comment explaining what rule it would use and what failure mode it would have that the cell and robot substrates do not.

4. **In one sentence:** What does this simulation demonstrate about substrate independence that you could not demonstrate with words alone?

## Your Answers

*Double-click this cell to edit. Write your answers here.*

1. 

2. 

3. 

4. 